# Proyecto Final — Airbnb: Madrid, Mallorca, Valencia, Sevilla, Milán
## Notebook 4: Visualización de Datos

**Objetivo:** Visualizar los patrones más relevantes del dataset. Cada figura incluye una interpretación analítica y se guarda en `../reports/figures/` para el informe y el dashboard.

> **Nota sobre Milán en gráficos de turismo:** Milán se excluye de las visualizaciones que comparan presión turística entre ciudades, para mantener comparabilidad territorial entre ciudades del mismo país. Se señala explícitamente en cada gráfico afectado.

**Fuentes de datos:** Inside Airbnb (anuncios Airbnb) + Eurostat `tour_occ_nin3` (pernoctaciones turísticas)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../reports/figures', exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white',
                     'axes.facecolor': 'white', 'font.size': 11})

CITY_COLORS = {
    'Madrid':   '#E63946',
    'Mallorca': '#2A9D8F',
    'Milan':    '#457B9D',
    'Sevilla':  '#F4A261',
    'Valencia': '#8338EC'
}
CITIES = list(CITY_COLORS.keys())
SPAIN  = ['Madrid', 'Mallorca', 'Valencia', 'Sevilla']

df = pd.read_csv('../data/processed/listings_final.csv', low_memory=False)
df_spain = df[df['city'].isin(SPAIN)].copy()

print(f'Dataset: {df.shape[0]:,} filas | Figuras → ../reports/figures/')

## 1. Distribución del precio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma recortado a ≤600€ para visualización
axes[0].hist(df[df['price'] <= 600]['price'], bins=70,
             color='#457B9D', edgecolor='white', linewidth=0.4)
axes[0].axvline(df['price'].mean(),   color='#E63946', lw=2, ls='--',
                label=f'Media: {df["price"].mean():.0f}€')
axes[0].axvline(df['price'].median(), color='#2A9D8F', lw=2, ls='--',
                label=f'Mediana: {df["price"].median():.0f}€')
axes[0].set_xlabel('Precio (€/noche)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del precio (recortado a ≤600€)')
axes[0].legend()

# KDE por ciudad, recortado a ≤600€
for city, color in CITY_COLORS.items():
    sub = df[(df['city'] == city) & (df['price'] <= 600)]['price']
    sub.plot.kde(ax=axes[1], label=city, color=color, linewidth=2)
axes[1].set_xlim(0, 600)
axes[1].set_xlabel('Precio (€/noche)')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Densidad de precio por ciudad (KDE, ≤600€)')
axes[1].legend(title='Ciudad')

plt.tight_layout()
plt.savefig('../reports/figures/01_distribucion_precio.png', bbox_inches='tight')
plt.show()
print('Guardada: 01_distribucion_precio.png')

> La distribución presenta fuerte asimetría a la derecha: la media (178€) supera ampliamente la mediana (130€). Madrid, Milán y Valencia concentran sus precios entre 50-200€. Mallorca muestra una distribución mucho más amplia y desplazada a la derecha, confirmando su mercado vacacional de alto precio.

## 2. Precio por ciudad — precio bruto y precio por persona

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precio mediano por ciudad
mediana = df.groupby('city')['price'].median().sort_values(ascending=False)
media   = df.groupby('city')['price'].mean()
colors  = [CITY_COLORS[c] for c in mediana.index]
bars = axes[0].bar(mediana.index, mediana.values, color=colors, edgecolor='white', width=0.6)
axes[0].scatter(mediana.index, [media[c] for c in mediana.index],
                color='black', zorder=5, s=60, marker='D', label='Media')
for bar, val in zip(bars, mediana.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 f'{val:.0f}€', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('Precio mediano por ciudad\n(diamante = media)')
axes[0].set_ylabel('€/noche')
axes[0].legend()

# Precio por persona — lectura más equitativa entre ciudades
pp_mediana = df.groupby('city')['price_per_person'].median().sort_values(ascending=False)
colors_pp  = [CITY_COLORS[c] for c in pp_mediana.index]
bars2 = axes[1].bar(pp_mediana.index, pp_mediana.values, color=colors_pp, edgecolor='white', width=0.6)
for bar, val in zip(bars2, pp_mediana.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.0f}€', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_title('Precio mediano por persona por ciudad\n(€/persona/noche)')
axes[1].set_ylabel('€/persona/noche')

plt.tight_layout()
plt.savefig('../reports/figures/02_precio_ciudad.png', bbox_inches='tight')
plt.show()
print('Guardada: 02_precio_ciudad.png')

> Mallorca lidera en precio bruto mediano (239€) y también en precio por persona. Al normalizar por capacidad, la brecha entre ciudades se reduce: Madrid y Milán tienen precios por persona similares, lo que sugiere que Madrid compensa su mayor precio bruto con alojamientos de mayor capacidad. Valencia es consistentemente la ciudad más asequible en ambas métricas.

## 3. Precio vs variables predictoras

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Precio mediano vs accommodates
acc = df[df['accommodates'] <= 12].groupby('accommodates')['price'].median().reset_index()
axes[0].plot(acc['accommodates'], acc['price'],
             marker='o', color='#E63946', linewidth=2.5, markersize=8)
axes[0].fill_between(acc['accommodates'], acc['price'], alpha=0.1, color='#E63946')
axes[0].set_title('Precio mediano vs capacidad\n(accommodates ≤ 12 personas)')
axes[0].set_xlabel('Nº de personas')
axes[0].set_ylabel('Precio mediano (€/noche)')
axes[0].set_xticks(range(1, 13))

# Precio mediano vs review_scores_rating (tramos)
bins_r = pd.cut(df['review_scores_rating'], bins=8)
rating_price = df.groupby(bins_r, observed=True)['price'].median().reset_index()
axes[1].bar(range(len(rating_price)), rating_price['price'],
            color='#457B9D', edgecolor='white')
axes[1].set_xticks(range(len(rating_price)))
axes[1].set_xticklabels(
    [f'{iv.left:.1f}' for iv in rating_price['review_scores_rating']],
    rotation=45, ha='right')
axes[1].set_title('Precio mediano vs valoración\n(review_scores_rating)')
axes[1].set_xlabel('Valoración (tramos)')
axes[1].set_ylabel('Precio mediano (€/noche)')

# Precio mediano vs availability_365 (tramos)
bins_a = pd.cut(df['availability_365'], bins=8)
av_price = df.groupby(bins_a, observed=True)['price'].median().reset_index()
axes[2].bar(range(len(av_price)), av_price['price'],
            color='#2A9D8F', edgecolor='white')
axes[2].set_xticks(range(len(av_price)))
axes[2].set_xticklabels(
    [f'{iv.left:.0f}' for iv in av_price['availability_365']],
    rotation=45, ha='right')
axes[2].set_title('Precio mediano vs disponibilidad\n(availability_365)')
axes[2].set_xlabel('Días disponibles (tramos)')
axes[2].set_ylabel('Precio mediano (€/noche)')

plt.tight_layout()
plt.savefig('../reports/figures/03_precio_vs_predictores.png', bbox_inches='tight')
plt.show()
print('Guardada: 03_precio_vs_predictores.png')

> La capacidad (`accommodates`) muestra una relación positiva clara y progresiva con el precio: el predictor más potente. La valoración no muestra relación lineal clara — el precio no depende principalmente de las reseñas. La disponibilidad muestra relación inversa: a menor disponibilidad, mayor precio mediano, coherente con mayor ocupación real o alquileres de temporada.

## 4. Precio mediano por ciudad y tipo de alojamiento

In [ ]:
pivot = df.groupby(['city','room_type'])['price'].median().unstack()

fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(kind='bar', ax=ax,
           color=['#E63946','#457B9D','#2A9D8F','#F4A261'],
           edgecolor='white', linewidth=0.5, width=0.75)
ax.set_title('Precio mediano por ciudad y tipo de alojamiento')
ax.set_xlabel('')
ax.set_ylabel('€/noche')
ax.set_xticklabels(pivot.index, rotation=0)
ax.legend(title='Tipo', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig('../reports/figures/04_precio_ciudad_roomtype.png', bbox_inches='tight')
plt.show()
print('Guardada: 04_precio_ciudad_roomtype.png')

> Mallorca presenta precios superiores en todos los tipos. En Sevilla, los `Hotel room` (232€) superan al `Entire home/apt` (131€), lo que indica que los hoteles sevillanos en Airbnb compiten en el segmento premium. Valencia es sistemáticamente la ciudad más asequible en todos los tipos.

## 5. Superhost: precio y valoración

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sh_price = df.groupby(['city','host_is_superhost'])['price'].median().unstack()
sh_price.columns = ['No Superhost','Superhost']
sh_price.plot(kind='bar', ax=axes[0], color=['#B0BEC5','#E63946'], edgecolor='white')
axes[0].set_title('Precio mediano: Superhost vs No Superhost')
axes[0].set_ylabel('€/noche')
axes[0].set_xlabel('')
axes[0].set_xticklabels(sh_price.index, rotation=0)
axes[0].legend()

sh_rating = df.groupby(['city','host_is_superhost'])['review_scores_rating'].mean().unstack()
sh_rating.columns = ['No Superhost','Superhost']
sh_rating.plot(kind='bar', ax=axes[1], color=['#B0BEC5','#E63946'], edgecolor='white')
axes[1].set_title('Valoración media: Superhost vs No Superhost')
axes[1].set_ylabel('Puntuación')
axes[1].set_xlabel('')
axes[1].set_xticklabels(sh_rating.index, rotation=0)
axes[1].set_ylim(4.0, 5.0)
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/05_superhost.png', bbox_inches='tight')
plt.show()
print('Guardada: 05_superhost.png')

> Los superhosts presentan precios medianos y valoraciones más altos en todas las ciudades. La diferencia de precio es más pronunciada en Mallorca y Milán. La diferencia de valoración es consistente pero pequeña, lo que confirma el análisis estadístico: la asociación existe pero la magnitud práctica es moderada.

## 6. Top 5 barrios más caros por ciudad

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5))

for idx, city in enumerate(CITIES):
    subset = df[df['city'] == city]
    top5 = (subset.groupby('neighbourhood_cleansed')['price']
            .median().sort_values(ascending=False).head(5))
    
    # Truncar nombres largos para legibilidad
    top5.index = [name[:20] + '…' if len(name) > 20 else name for name in top5.index]
    
    top5.plot(kind='barh', ax=axes[idx], color=CITY_COLORS[city], edgecolor='white')
    axes[idx].set_title(f'{city}', fontweight='bold', fontsize=11)
    axes[idx].set_xlabel('€/noche (mediana)', fontsize=9)
    axes[idx].invert_yaxis()
    axes[idx].tick_params(axis='y', labelsize=8)

plt.suptitle('Top 5 barrios más caros por ciudad (precio mediano)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/06_top_barrios.png', bbox_inches='tight')
plt.show()
print('Guardada: 06_top_barrios.png')

> En Madrid, los barrios prime del centro histórico y Salamanca lideran en precio. En Mallorca, las zonas costeras exclusivas superan ampliamente al resto. En Milán, los barrios de diseño y moda concentran los precios más altos. En Sevilla y Valencia, la concentración turística en el casco histórico eleva los precios de esas zonas.

## 7. Disponibilidad por ciudad

In [ ]:
disp = df.groupby('city')['availability_365'].mean().sort_values(ascending=False)
colors = [CITY_COLORS[c] for c in disp.index]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(disp.index, disp.values, color=colors, edgecolor='white', width=0.6)
ax.axhline(180, color='#E63946', lw=1.5, ls='--', alpha=0.7, label='Umbral alta disponibilidad (180d)')
for bar, val in zip(bars, disp.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:.0f}d', ha='center', va='bottom', fontweight='bold')
ax.set_title('Disponibilidad media anual por ciudad')
ax.set_ylabel('Días disponibles al año')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/07_disponibilidad.png', bbox_inches='tight')
plt.show()
print('Guardada: 07_disponibilidad.png')

> Mallorca tiene la menor disponibilidad media, coherente con su estacionalidad estival. Las ciudades que superan el umbral de 180 días (Milán, Valencia, Madrid) tienen mercados más activos durante todo el año. Esta métrica es consistente con la asociación encontrada entre baja disponibilidad y mayor precio.

## 8. Mapa de correlaciones

In [ ]:
cols_corr = [
    'price','accommodates','bathrooms','bedrooms','beds',
    'minimum_nights','availability_365','number_of_reviews',
    'review_scores_rating','review_scores_location','reviews_per_month'
]
corr_matrix = df[cols_corr].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap — visión global de relaciones entre variables
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            ax=axes[0], linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[0].set_title('Mapa de correlaciones (Pearson)\nvisión global', fontsize=12)

# Barplot — importancia relativa con price
corr_price = corr_matrix['price'].drop('price').sort_values(key=abs, ascending=False)
colors_bar = ['#E63946' if v > 0 else '#457B9D' for v in corr_price.values]
corr_price.plot(kind='barh', ax=axes[1], color=colors_bar, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlación de cada variable con price\n(rojo = positiva, azul = negativa)', fontsize=12)
axes[1].set_xlabel('Coeficiente de Pearson')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../reports/figures/08_correlaciones.png', bbox_inches='tight')
plt.show()
print('Guardada: 08_correlaciones.png')

> El heatmap muestra la visión global de relaciones. El barplot complementario destaca que `accommodates`, `bathrooms` y `bedrooms` son los predictores positivos más fuertes del precio. `reviews_per_month` y `number_of_reviews` tienen correlación negativa — los alojamientos baratos rotan más y acumulan más reseñas. Ambos gráficos se incluyen al ser complementarios: el heatmap para visión global, el barplot para el ranking de predictores.

## 9. Turismo (Eurostat) vs precio Airbnb — solo ciudades españolas

> **Nota:** Se excluye Milán de esta visualización para mantener comparabilidad territorial. La comparativa se realiza únicamente entre las cuatro ciudades españolas del dataset.

In [ ]:
turismo = df_spain.groupby('city').agg(
    precio_mediano=('price','median'),
    tourism_pressure=('tourism_pressure','first'),
    n_anuncios=('price','count')
).reset_index()

fig, ax = plt.subplots(figsize=(9, 6))
for _, row in turismo.iterrows():
    color = CITY_COLORS[row['city']]
    ax.scatter(row['tourism_pressure'], row['precio_mediano'],
               s=row['n_anuncios'] / 5, color=color, alpha=0.85,
               edgecolors='white', linewidth=1.5, zorder=3)
    ax.annotate(row['city'],
                xy=(row['tourism_pressure'], row['precio_mediano']),
                xytext=(10, 6), textcoords='offset points',
                fontsize=12, fontweight='bold', color=color)

ax.set_xlabel('Presión turística — pernoctaciones Eurostat (normalizado 0–1)', fontsize=11)
ax.set_ylabel('Precio mediano Airbnb (€/noche)', fontsize=11)
ax.set_title('Presión turística vs precio Airbnb\nCiudades españolas (tamaño = nº de anuncios)',
             fontsize=12)
ax.grid(True, alpha=0.3)
ax.text(0.02, 0.02, 'Milán excluida para mantener comparabilidad territorial (análisis restringido a ciudades españolas)',
        transform=ax.transAxes, fontsize=9, color='gray', style='italic')
plt.tight_layout()
plt.savefig('../reports/figures/09_turismo_vs_precio_spain.png', bbox_inches='tight')
plt.show()
print('Guardada: 09_turismo_vs_precio_spain.png')

> En las ciudades españolas, Mallorca combina la mayor presión turística (1.0) con el precio mediano más alto (239€). Sin embargo, Madrid, con mayor turismo que Sevilla y Valencia, presenta un precio mediano similar al de estas ciudades, lo que indica que la abundante oferta de alojamiento en la capital modera los precios a pesar del volumen turístico.

## Resumen de figuras generadas

| Figura | Descripción | Hallazgo principal |
|---|---|---|
| 01_distribucion_precio | Histograma + KDE por ciudad | Fuerte asimetría derecha; Mallorca con cola más amplia |
| 02_precio_ciudad | Precio mediano + precio por persona | Mallorca lidera en ambas métricas; Valencia más asequible |
| 03_precio_vs_predictores | Price vs accommodates, rating, disponibilidad | Capacidad: relación positiva clara y progresiva |
| 04_precio_ciudad_roomtype | Precio mediano ciudad × tipo | Sevilla: hoteles más caros que apartamentos enteros |
| 05_superhost | Precio y valoración Superhost vs No | Superhosts: precios y valoraciones consistentemente superiores |
| 06_top_barrios | Top 5 barrios más caros por ciudad | Concentración de precios en zonas prime de cada ciudad |
| 07_disponibilidad | Disponibilidad media por ciudad | Mallorca: menor disponibilidad (estacionalidad) |
| 08_correlaciones | Heatmap + barplot correlaciones | Tamaño del alojamiento = predictor más fuerte del precio |
| 09_turismo_vs_precio_spain | Turismo Eurostat vs precio (solo España) | Mayor turismo → mayor precio, con excepción de Madrid |

---

**Conclusión de las visualizaciones:** Los gráficos confirman de forma consistente que el precio en Airbnb está principalmente determinado por las características físicas del alojamiento (tamaño, capacidad) y su localización (ciudad, barrio). Los factores reputacionales — valoración media, número de reseñas — tienen una influencia visual menor y no muestran una relación clara y progresiva con el precio. Mallorca destaca sistemáticamente como mercado diferenciado en todas las dimensiones analizadas.